# 05B — Sentiment Model Training for Entity-Conditioned AI Impact Analysis

Pipeline stage:

- load the sentiment training dataset from Stage 05A
- clean and validate label space
- create a group-aware train / validation split to reduce entity leakage
- train a transformer classifier for 3-way entity-conditioned sentiment
- evaluate overall performance and by entity type
- export model artifacts, metrics, reports, predictions, and hard cases

Input directory

`output/05_sentiment_dataset/`

- `sentiment_training_data.parquet`
- `sentiment_label_summary.csv`
- `sentiment_labeling_report.json`

Output directory

`output/05_sentiment_model/`

- `train_split.parquet`
- `valid_split.parquet`
- `label_mapping.json`
- `training_run_config.json`
- `best_model/`
- `training_metrics.json`
- `validation_metrics.json`
- `classification_report.json`
- `classification_report_by_type.json`
- `confusion_matrix.csv`
- `validation_predictions.parquet`
- `validation_hard_cases.parquet`

In [1]:
!pip -q uninstall -y transformers accelerate datasets gcsfs fsspec tokenizers
!pip -q install \
    transformers==4.45.1 \
    accelerate==0.34.2 \
    datasets==2.21.0 \
    evaluate==0.4.3 \
    scikit-learn==1.5.2 \
    pyarrow==17.0.0 \
    fsspec==2024.6.1

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 27.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 324.4/324.4 kB 41.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.3/527.3 kB 57.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.9/12.9 MB 178.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.9/39.9 MB 74.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.6/177.6 kB 26.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 70.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 170.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.37.0 requires gc

## Environment Setup

In [2]:
import os
import gc
import re
import json
import random
import hashlib
import warnings
from typing import Dict, Any, List, Tuple, Optional

import numpy as np
import pandas as pd

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score,
    accuracy_score,
    precision_recall_fscore_support,
)

import torch
from datasets import Dataset, DatasetDict

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    set_seed,
)

from google.colab import drive
drive.mount("/content/drive")

warnings.filterwarnings("ignore")

try:
    from IPython.display import display
except Exception:
    def display(x):
        print(x)

Mounted at /content/drive


## Paths

In [3]:
BASE_DIR = "/content/drive/MyDrive/NLP_FINAL_PROJECT_Tom_Chen"

IN_DIR_05A = f"{BASE_DIR}/output/05_sentiment_dataset"
OUT_DIR_05B = f"{BASE_DIR}/output/05_sentiment_model"

SENTIMENT_TRAINING_DATA_PARQUET = f"{IN_DIR_05A}/sentiment_training_data.parquet"
SENTIMENT_LABEL_SUMMARY_CSV = f"{IN_DIR_05A}/sentiment_label_summary.csv"
SENTIMENT_LABELING_REPORT_JSON = f"{IN_DIR_05A}/sentiment_labeling_report.json"

TRAIN_SPLIT_PARQUET = f"{OUT_DIR_05B}/train_split.parquet"
VALID_SPLIT_PARQUET = f"{OUT_DIR_05B}/valid_split.parquet"
LABEL_MAPPING_JSON = f"{OUT_DIR_05B}/label_mapping.json"
TRAINING_RUN_CONFIG_JSON = f"{OUT_DIR_05B}/training_run_config.json"
TRAINING_METRICS_JSON = f"{OUT_DIR_05B}/training_metrics.json"
VALIDATION_METRICS_JSON = f"{OUT_DIR_05B}/validation_metrics.json"
CLASSIFICATION_REPORT_JSON = f"{OUT_DIR_05B}/classification_report.json"
CLASSIFICATION_REPORT_BY_TYPE_JSON = f"{OUT_DIR_05B}/classification_report_by_type.json"
CONFUSION_MATRIX_CSV = f"{OUT_DIR_05B}/confusion_matrix.csv"
VALIDATION_PREDICTIONS_PARQUET = f"{OUT_DIR_05B}/validation_predictions.parquet"
VALIDATION_HARD_CASES_PARQUET = f"{OUT_DIR_05B}/validation_hard_cases.parquet"
BEST_MODEL_DIR = f"{OUT_DIR_05B}/best_model"

os.makedirs(OUT_DIR_05B, exist_ok=True)

assert os.path.exists(SENTIMENT_TRAINING_DATA_PARQUET), f"Missing: {SENTIMENT_TRAINING_DATA_PARQUET}"

print("SENTIMENT_TRAINING_DATA_PARQUET:", SENTIMENT_TRAINING_DATA_PARQUET)
print("OUT_DIR_05B:", OUT_DIR_05B)

SENTIMENT_TRAINING_DATA_PARQUET: /content/drive/MyDrive/NLP_FINAL_PROJECT_Tom_Chen/output/05_sentiment_dataset/sentiment_training_data.parquet
OUT_DIR_05B: /content/drive/MyDrive/NLP_FINAL_PROJECT_Tom_Chen/output/05_sentiment_model


## Runtime and Configuration

In [4]:
SEED = 42
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

MODEL_NAME = "roberta-large"

LABEL_COL = "sentiment_label"
TEXT_COL = "context_text"
ENTITY_COL = "canonical_entity"
TYPE_COL = "final_type"
GROUP_SPLIT_COL = "entity_group_id"
MODEL_INPUT_COL = "model_input"

ALLOWED_LABELS = ["positive", "negative", "mixed_or_unclear"]
LABEL2ID = {x: i for i, x in enumerate(ALLOWED_LABELS)}
ID2LABEL = {i: x for x, i in LABEL2ID.items()}

VALID_ENTITY_TYPES = {"company", "technology", "government_institution", "person"}

MIN_CONTEXT_CHARS = 40
MAX_CONTEXT_CHARS = 1400
MIN_ENTITY_LEN = 2

VALID_SIZE = 0.18
MIN_VALID_ROWS = 1200

MAX_LENGTH = 384

NUM_EPOCHS = 4
LEARNING_RATE = 1.5e-5
WEIGHT_DECAY = 0.03
TRAIN_BATCH_SIZE = 16
EVAL_BATCH_SIZE = 32
GRAD_ACCUM_STEPS = 2
WARMUP_RATIO = 0.10
EARLY_STOPPING_PATIENCE = 2
LABEL_SMOOTHING = 0.05
MAX_GRAD_NORM = 1.0

SAVE_TOTAL_LIMIT = 2
LOGGING_STEPS = 50
EVALUATION_STRATEGY = "epoch"
SAVE_STRATEGY = "epoch"
LOAD_BEST_MODEL_AT_END = True
METRIC_FOR_BEST_MODEL = "macro_f1"
GREATER_IS_BETTER = True

USE_CLASS_WEIGHTS = True
USE_ENTITY_CONDITIONED_INPUT = True
TEXT_TEMPLATE_VERSION = "v2_entity_conditioned"

USE_FP16 = False
USE_BF16 = torch.cuda.is_available()

HARD_CASE_TOP_N = 1000
REUSE_SPLITS_IF_PRESENT = False

run_config = {
    "seed": SEED,
    "model_name": MODEL_NAME,
    "label_col": LABEL_COL,
    "text_col": TEXT_COL,
    "entity_col": ENTITY_COL,
    "type_col": TYPE_COL,
    "group_split_col": GROUP_SPLIT_COL,
    "model_input_col": MODEL_INPUT_COL,
    "allowed_labels": ALLOWED_LABELS,
    "valid_entity_types": sorted(VALID_ENTITY_TYPES),
    "min_context_chars": MIN_CONTEXT_CHARS,
    "max_context_chars": MAX_CONTEXT_CHARS,
    "min_entity_len": MIN_ENTITY_LEN,
    "valid_size": VALID_SIZE,
    "min_valid_rows": MIN_VALID_ROWS,
    "max_length": MAX_LENGTH,
    "num_epochs": NUM_EPOCHS,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "train_batch_size": TRAIN_BATCH_SIZE,
    "eval_batch_size": EVAL_BATCH_SIZE,
    "grad_accum_steps": GRAD_ACCUM_STEPS,
    "warmup_ratio": WARMUP_RATIO,
    "early_stopping_patience": EARLY_STOPPING_PATIENCE,
    "label_smoothing": LABEL_SMOOTHING,
    "max_grad_norm": MAX_GRAD_NORM,
    "evaluation_strategy": EVALUATION_STRATEGY,
    "save_strategy": SAVE_STRATEGY,
    "load_best_model_at_end": LOAD_BEST_MODEL_AT_END,
    "metric_for_best_model": METRIC_FOR_BEST_MODEL,
    "greater_is_better": GREATER_IS_BETTER,
    "use_class_weights": USE_CLASS_WEIGHTS,
    "use_entity_conditioned_input": USE_ENTITY_CONDITIONED_INPUT,
    "text_template_version": TEXT_TEMPLATE_VERSION,
    "use_fp16": USE_FP16,
    "use_bf16": USE_BF16,
    "reuse_splits_if_present": REUSE_SPLITS_IF_PRESENT,
    "hard_case_top_n": HARD_CASE_TOP_N,
}

with open(TRAINING_RUN_CONFIG_JSON, "w", encoding="utf-8") as f:
    json.dump(run_config, f, indent=2)

print("wrote:", TRAINING_RUN_CONFIG_JSON)
print("torch.cuda.is_available():", torch.cuda.is_available())
if torch.cuda.is_available():
    print("cuda device:", torch.cuda.get_device_name(0))

wrote: /content/drive/MyDrive/NLP_FINAL_PROJECT_Tom_Chen/output/05_sentiment_model/training_run_config.json
torch.cuda.is_available(): True
cuda device: NVIDIA H100 80GB HBM3


## Utility Functions

In [5]:
WS_RE = re.compile(r"\s+")

def normalize_spaces(text: str) -> str:
    return WS_RE.sub(" ", str(text or "")).strip()

def safe_json_dump(obj: Any, path: str) -> None:
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)

def sha1_text(text: str) -> str:
    return hashlib.sha1(str(text).encode("utf-8")).hexdigest()

def truncate_text(text: str, max_chars: int) -> str:
    text = normalize_spaces(text)
    if len(text) <= max_chars:
        return text
    return text[:max_chars].rstrip() + " ..."

def build_model_input(row: pd.Series) -> str:
    entity = normalize_spaces(row.get(ENTITY_COL, ""))
    entity_type = normalize_spaces(row.get(TYPE_COL, ""))
    title = normalize_spaces(row.get("title", ""))
    domain = normalize_spaces(row.get("domain", ""))
    context = truncate_text(row.get(TEXT_COL, ""), MAX_CONTEXT_CHARS)

    if USE_ENTITY_CONDITIONED_INPUT:
        parts = [
            f"Target entity: {entity}",
            f"Entity type: {entity_type}",
        ]
        if title:
            parts.append(f"Document title: {title}")
        if domain:
            parts.append(f"Source domain: {domain}")
        parts.append(
            "Instruction: classify the sentiment of AI-related impact on the target entity "
            "in this context as positive, negative, or mixed_or_unclear."
        )
        parts.append(f"Context: {context}")
        return "\n".join(parts)

    return context

def assign_group_bucket(entity_stats: pd.DataFrame) -> pd.DataFrame:
    out = entity_stats.copy()
    out["bucket"] = np.select(
        [
            out["n_docs"] >= 300,
            out["n_docs"] >= 80,
            out["n_docs"] >= 20,
        ],
        [
            "head",
            "mid",
            "tail",
        ],
        default="tail"
    )
    return out

def group_aware_stratified_split(
    df: pd.DataFrame,
    group_col: str,
    valid_size: float,
    seed: int = SEED,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    group_stats = (
        df.groupby(group_col, dropna=False)
        .agg(
            n_rows=("sample_id", "size"),
            n_docs=("doc_id", "nunique"),
            dominant_type=(TYPE_COL, lambda x: x.mode().iloc[0] if len(x.mode()) else x.iloc[0]),
            dominant_label=(LABEL_COL, lambda x: x.mode().iloc[0] if len(x.mode()) else x.iloc[0]),
        )
        .reset_index()
    )

    group_stats = assign_group_bucket(group_stats)
    group_stats["stratum"] = (
        group_stats["dominant_type"].astype(str)
        + "|||"
        + group_stats["dominant_label"].astype(str)
        + "|||"
        + group_stats["bucket"].astype(str)
    )

    valid_groups = []

    for _, sub in group_stats.groupby("stratum", dropna=False):
        sub = sub.sample(frac=1.0, random_state=seed).reset_index(drop=True)
        target_n = max(1, int(round(len(sub) * valid_size)))
        valid_groups.extend(sub.head(target_n)[group_col].tolist())

    valid_groups = set(valid_groups)

    valid_df = df[df[group_col].isin(valid_groups)].copy()
    train_df = df[~df[group_col].isin(valid_groups)].copy()

    if len(valid_df) < MIN_VALID_ROWS:
        remaining = group_stats[~group_stats[group_col].isin(valid_groups)].copy()
        remaining = remaining.sample(frac=1.0, random_state=seed).reset_index(drop=True)

        for group_value in remaining[group_col].tolist():
            extra = df[df[group_col] == group_value]
            valid_df = pd.concat([valid_df, extra], ignore_index=True)
            train_df = train_df[train_df[group_col] != group_value].copy()
            if len(valid_df) >= MIN_VALID_ROWS:
                break

    return train_df.reset_index(drop=True), valid_df.reset_index(drop=True)

def build_class_weights(label_ids: pd.Series) -> torch.Tensor:
    counts = label_ids.value_counts().sort_index()
    counts = counts.reindex(range(len(ALLOWED_LABELS)), fill_value=1)
    weights = len(label_ids) / (len(ALLOWED_LABELS) * counts.values.astype(float))
    weights = weights / weights.mean()
    return torch.tensor(weights, dtype=torch.float32)

def compute_metrics_from_logits(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    macro_f1 = f1_score(labels, preds, average="macro")
    weighted_f1 = f1_score(labels, preds, average="weighted")
    acc = accuracy_score(labels, preds)

    precision_macro, recall_macro, _, _ = precision_recall_fscore_support(
        labels, preds, average="macro", zero_division=0
    )

    return {
        "accuracy": float(acc),
        "macro_f1": float(macro_f1),
        "weighted_f1": float(weighted_f1),
        "macro_precision": float(precision_macro),
        "macro_recall": float(recall_macro),
    }

class WeightedTrainer(Trainer):
    def __init__(self, class_weights: Optional[torch.Tensor] = None, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")

        loss_fct = torch.nn.CrossEntropyLoss(
            weight=self.class_weights.to(logits.device) if self.class_weights is not None else None,
            label_smoothing=LABEL_SMOOTHING,
        )
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

## Load Training Data

In [6]:
training_data = pd.read_parquet(SENTIMENT_TRAINING_DATA_PARQUET)

print("training_data raw shape:", training_data.shape)
display(training_data.head(5))

required_cols = {
    "sample_id",
    "doc_id",
    "block_id",
    "domain",
    "title",
    "canonical_entity",
    "final_type",
    "context_text",
    "sentiment_label",
    "confidence",
}
missing_cols = required_cols - set(training_data.columns)
assert not missing_cols, f"Missing required columns: {missing_cols}"

training_data raw shape: (8999, 21)


,sample_id,sample_source,doc_id,block_id,date,time_bucket,domain,title,url,canonical_entity,...,final_type,context_text,n_docs,n_rows,n_domains,confidence_mean,entity_doc_bucket,sentiment_label,confidence,reason_short
0,93375||8||jordan||government_institution,government_institution_stratified_context_pool,93375,8,2023-06-05,2023Q2,telecom.economictimes.indiatimes.com,OpenAI CEO sees 'huge' Israeli role in reducin...,https://telecom.economictimes.indiatimes.com/a...,Jordan,...,government_institution,After crisscrossing Europe last month meeting ...,90,143,70,0.797979,mid,mixed_or_unclear,0.65,Context describes travel plans without specify...
1,94750||9||quantum computing||technology,technology_stratified_context_pool,94750,9,2022-06-07,2022Q2,newsbytes.ph,"After AI and blockchain, IBM tags quantum comp...",https://newsbytes.ph/2022/06/06/after-ai-and-b...,Quantum Computing,...,technology,IBM’s director of Quantum Infrastructure Jerry...,658,1315,291,0.873885,head,positive,0.85,IBM emphasizes quantum computing's exponential...
2,23320||18||quantum computing||technology,technology_stratified_context_pool,23320,18,2025-02-10,2025Q1,einpresswire.com,LEAP 2025 Opens with Announcement of Record-br...,https://www.einpresswire.com/article/784621089...,Quantum Computing,...,technology,“A breakthrough I think is only about three-to...,658,1315,291,0.873885,head,positive,0.85,Quantum computing described as breakthrough en...
3,134159||23||dell||company,company_stratified_context_pool,134159,23,2024-05-28,2024Q2,computerweekly.com,Executive Interview: Why Dell wants to be your...,https://www.computerweekly.com/news/366586034/...,Dell,...,company,"At Dell Technologies World in Las Vegas, artif...",670,2256,278,0.850091,head,positive,0.85,"Dell positions AI as central to its strategy, ..."
4,190378||7||quantum computing||technology,technology_stratified_context_pool,190378,7,2024-11-04,2024Q4,businessday.ng,Adopting quantum computing and artificial inte...,https://businessday.ng/opinion/article/adoptin...,Quantum Computing,...,technology,Healthcare: Quantum computing can enhance drug...,658,1315,291,0.873885,head,positive,0.85,"Quantum computing enhances healthcare, agricul..."


## Clean and Validate Training Data

In [7]:
df = training_data.copy()

df["sample_id"] = df["sample_id"].astype(str)
df["canonical_entity"] = df["canonical_entity"].fillna("").astype(str).map(normalize_spaces)
df["final_type"] = df["final_type"].fillna("").astype(str).map(normalize_spaces)
df["context_text"] = df["context_text"].fillna("").astype(str).map(normalize_spaces)
df["title"] = df["title"].fillna("").astype(str).map(normalize_spaces)
df["domain"] = df["domain"].fillna("").astype(str).map(normalize_spaces)
df["sentiment_label"] = df["sentiment_label"].fillna("").astype(str).map(normalize_spaces)
df["confidence"] = pd.to_numeric(df["confidence"], errors="coerce").clip(0.0, 1.0)

df = df[df["final_type"].isin(VALID_ENTITY_TYPES)].copy()
df = df[df["sentiment_label"].isin(ALLOWED_LABELS)].copy()
df = df[df["canonical_entity"].str.len() >= MIN_ENTITY_LEN].copy()
df = df[df["context_text"].str.len() >= MIN_CONTEXT_CHARS].copy()

df = df.drop_duplicates("sample_id").reset_index(drop=True)

df[MODEL_INPUT_COL] = df.apply(build_model_input, axis=1)
df[MODEL_INPUT_COL] = df[MODEL_INPUT_COL].map(normalize_spaces)

df["label_id"] = df["sentiment_label"].map(LABEL2ID).astype(int)
df[GROUP_SPLIT_COL] = (
    df["canonical_entity"].astype(str).str.lower()
    + "|||"
    + df["final_type"].astype(str).str.lower()
)
df["input_hash"] = df[MODEL_INPUT_COL].map(sha1_text)

df = df.drop_duplicates("input_hash").reset_index(drop=True)

print("training_data cleaned shape:", df.shape)
print("\nrows by final_type:")
display(df["final_type"].value_counts(dropna=False))
print("\nrows by sentiment_label:")
display(df["sentiment_label"].value_counts(dropna=False))
print("\nconfidence summary:")
display(df["confidence"].describe(percentiles=[0.01, 0.1, 0.5, 0.9, 0.99]))

print("\nexample model inputs:")
display(df[[ENTITY_COL, TYPE_COL, LABEL_COL, MODEL_INPUT_COL]].head(3))

training_data cleaned shape: (8876, 25)

rows by final_type:


,count
final_type,
company,3164
technology,3149
government_institution,1383
person,1180



rows by sentiment_label:


,count
sentiment_label,
positive,5095
mixed_or_unclear,2632
negative,1149



confidence summary:


,confidence
count,8876.000000
mean,0.826009
std,0.060664
min,0.500000
1%,0.650000
10%,0.750000
50%,0.850000
90%,0.920000
99%,0.920000
max,0.950000



example model inputs:


,canonical_entity,final_type,sentiment_label,model_input
0,Jordan,government_institution,mixed_or_unclear,Target entity: Jordan Entity type: government_...
1,Quantum Computing,technology,positive,Target entity: Quantum Computing Entity type: ...
2,Quantum Computing,technology,positive,Target entity: Quantum Computing Entity type: ...


## Train / Validation Split

In [8]:
if REUSE_SPLITS_IF_PRESENT and os.path.exists(TRAIN_SPLIT_PARQUET) and os.path.exists(VALID_SPLIT_PARQUET):
    train_df = pd.read_parquet(TRAIN_SPLIT_PARQUET)
    valid_df = pd.read_parquet(VALID_SPLIT_PARQUET)
else:
    train_df, valid_df = group_aware_stratified_split(
        df=df,
        group_col=GROUP_SPLIT_COL,
        valid_size=VALID_SIZE,
        seed=SEED,
    )

    train_df.to_parquet(TRAIN_SPLIT_PARQUET, index=False)
    valid_df.to_parquet(VALID_SPLIT_PARQUET, index=False)

print("train_df shape:", train_df.shape)
print("valid_df shape:", valid_df.shape)
print("wrote:", TRAIN_SPLIT_PARQUET)
print("wrote:", VALID_SPLIT_PARQUET)

train_groups = set(train_df[GROUP_SPLIT_COL].unique().tolist())
valid_groups = set(valid_df[GROUP_SPLIT_COL].unique().tolist())
entity_leakage_n = len(train_groups & valid_groups)

print("entity leakage groups:", entity_leakage_n)

print("\ntrain labels:")
display(train_df["sentiment_label"].value_counts(dropna=False))
print("\nvalid labels:")
display(valid_df["sentiment_label"].value_counts(dropna=False))

print("\ntrain types:")
display(train_df["final_type"].value_counts(dropna=False))
print("\nvalid types:")
display(valid_df["final_type"].value_counts(dropna=False))

print("\ntrain label x type:")
display(pd.crosstab(train_df["final_type"], train_df["sentiment_label"]))

print("\nvalid label x type:")
display(pd.crosstab(valid_df["final_type"], valid_df["sentiment_label"]))

class_weights = build_class_weights(train_df["label_id"]) if USE_CLASS_WEIGHTS else None
print("\nclass weights:")
print(class_weights)

safe_json_dump(
    {
        "label2id": LABEL2ID,
        "id2label": ID2LABEL,
    },
    LABEL_MAPPING_JSON,
)
print("wrote:", LABEL_MAPPING_JSON)

train_df shape: (7251, 25)
valid_df shape: (1625, 25)
wrote: /content/drive/MyDrive/NLP_FINAL_PROJECT_Tom_Chen/output/05_sentiment_model/train_split.parquet
wrote: /content/drive/MyDrive/NLP_FINAL_PROJECT_Tom_Chen/output/05_sentiment_model/valid_split.parquet
entity leakage groups: 0

train labels:


,count
sentiment_label,
positive,4135
mixed_or_unclear,2160
negative,956



valid labels:


,count
sentiment_label,
positive,960
mixed_or_unclear,472
negative,193



train types:


...


train label x type:


sentiment_label,mixed_or_unclear,negative,positive
final_type,,,
company,582,261,1703
government_institution,433,254,466
person,385,198,377
technology,760,243,1589



valid label x type:


sentiment_label,mixed_or_unclear,negative,positive
final_type,,,
company,139,60,419
government_institution,85,40,105
person,97,43,80
technology,151,50,356



class weights:
tensor([0.4144, 1.7923, 0.7933])
wrote: /content/drive/MyDrive/NLP_FINAL_PROJECT_Tom_Chen/output/05_sentiment_model/label_mapping.json


## Tokenization

In [9]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

train_dataset = Dataset.from_pandas(
    train_df[["sample_id", MODEL_INPUT_COL, "label_id", "canonical_entity", "final_type", "sentiment_label"]],
    preserve_index=False,
)
valid_dataset = Dataset.from_pandas(
    valid_df[["sample_id", MODEL_INPUT_COL, "label_id", "canonical_entity", "final_type", "sentiment_label"]],
    preserve_index=False,
)

def tokenize_batch(batch: Dict[str, List[Any]]) -> Dict[str, Any]:
    enc = tokenizer(
        batch[MODEL_INPUT_COL],
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False,
    )
    enc["labels"] = batch["label_id"]
    return enc

dataset_dict = DatasetDict({
    "train": train_dataset,
    "valid": valid_dataset,
})

tokenized = dataset_dict.map(
    tokenize_batch,
    batched=True,
    remove_columns=dataset_dict["train"].column_names,
    desc="Tokenizing datasets"
)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

print(tokenized)

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Tokenizing datasets:   0%|          | 0/7251 [00:00<?, ? examples/s]

Tokenizing datasets:   0%|          | 0/1625 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 7251
    })
    valid: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 1625
    })
})


## Model Setup

In [10]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(ALLOWED_LABELS),
    id2label=ID2LABEL,
    label2id=LABEL2ID,
)

training_args = TrainingArguments(
    output_dir=OUT_DIR_05B,
    overwrite_output_dir=True,
    num_train_epochs=NUM_EPOCHS,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    warmup_ratio=WARMUP_RATIO,
    logging_steps=LOGGING_STEPS,
    evaluation_strategy=EVALUATION_STRATEGY,
    save_strategy=SAVE_STRATEGY,
    save_total_limit=SAVE_TOTAL_LIMIT,
    load_best_model_at_end=LOAD_BEST_MODEL_AT_END,
    metric_for_best_model=METRIC_FOR_BEST_MODEL,
    greater_is_better=GREATER_IS_BETTER,
    fp16=USE_FP16,
    bf16=USE_BF16,
    max_grad_norm=MAX_GRAD_NORM,
    report_to="none",
    dataloader_num_workers=4,
    dataloader_pin_memory=True,
    seed=SEED,
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["valid"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics_from_logits,
    class_weights=class_weights,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=EARLY_STOPPING_PATIENCE)],
)

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-large and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


## Train

In [11]:
train_result = trainer.train()
trainer.save_model(BEST_MODEL_DIR)
tokenizer.save_pretrained(BEST_MODEL_DIR)

train_metrics = dict(train_result.metrics)
train_metrics["best_model_checkpoint"] = trainer.state.best_model_checkpoint
train_metrics["best_metric"] = float(trainer.state.best_metric) if trainer.state.best_metric is not None else None
safe_json_dump(train_metrics, TRAINING_METRICS_JSON)

print("wrote:", BEST_MODEL_DIR)
print("wrote:", TRAINING_METRICS_JSON)
print(train_metrics)

Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Weighted F1,Macro Precision,Macro Recall
1,0.669700,0.631905,0.818462,0.783991,0.817291,0.773859,0.799093
2,0.517800,0.645534,0.830769,0.779141,0.821835,0.790925,0.814193
3,0.387400,0.595526,0.854154,0.824405,0.855214,0.807981,0.846728
4,0.302400,0.637985,0.849846,0.821355,0.849181,0.810235,0.837174


wrote: /content/drive/MyDrive/NLP_FINAL_PROJECT_Tom_Chen/output/05_sentiment_model/best_model
wrote: /content/drive/MyDrive/NLP_FINAL_PROJECT_Tom_Chen/output/05_sentiment_model/training_metrics.json
{'train_runtime': 213.5791, 'train_samples_per_second': 135.8, 'train_steps_per_second': 4.251, 'total_flos': 1.9849311397690748e+16, 'train_loss': 0.5173748763122222, 'epoch': 4.0, 'best_model_checkpoint': '/content/drive/MyDrive/NLP_FINAL_PROJECT_Tom_Chen/output/05_sentiment_model/checkpoint-681', 'best_metric': 0.8244049281569065}


## Validation Prediction and Metrics

In [12]:
pred_output = trainer.predict(tokenized["valid"])
val_logits = pred_output.predictions
val_label_ids = pred_output.label_ids
val_pred_ids = np.argmax(val_logits, axis=1)

valid_pred = valid_df.copy().reset_index(drop=True)
valid_pred["label_id"] = val_label_ids.astype(int)
valid_pred["pred_id"] = val_pred_ids.astype(int)
valid_pred["pred_label"] = valid_pred["pred_id"].map(ID2LABEL)
valid_pred["true_label"] = valid_pred["label_id"].map(ID2LABEL)

probs = torch.softmax(torch.tensor(val_logits), dim=1).cpu().numpy()
valid_pred["pred_confidence"] = probs.max(axis=1)

for label_name, label_id in LABEL2ID.items():
    valid_pred[f"prob_{label_name}"] = probs[:, label_id]

valid_pred["is_correct"] = valid_pred["true_label"] == valid_pred["pred_label"]

validation_metrics = {
    "accuracy": float(accuracy_score(val_label_ids, val_pred_ids)),
    "macro_f1": float(f1_score(val_label_ids, val_pred_ids, average="macro")),
    "weighted_f1": float(f1_score(val_label_ids, val_pred_ids, average="weighted")),
}

for label_name, label_id in LABEL2ID.items():
    y_true_bin = (val_label_ids == label_id).astype(int)
    y_pred_bin = (val_pred_ids == label_id).astype(int)
    p, r, f, _ = precision_recall_fscore_support(
        y_true_bin, y_pred_bin, average="binary", zero_division=0
    )
    validation_metrics[f"{label_name}_precision"] = float(p)
    validation_metrics[f"{label_name}_recall"] = float(r)
    validation_metrics[f"{label_name}_f1"] = float(f)

safe_json_dump(validation_metrics, VALIDATION_METRICS_JSON)

print("wrote:", VALIDATION_METRICS_JSON)
print(validation_metrics)

wrote: /content/drive/MyDrive/NLP_FINAL_PROJECT_Tom_Chen/output/05_sentiment_model/validation_metrics.json
{'accuracy': 0.8541538461538462, 'macro_f1': 0.8244049281569065, 'weighted_f1': 0.8552139050224518, 'positive_precision': 0.9298056155507559, 'positive_recall': 0.896875, 'positive_f1': 0.9130434782608695, 'negative_precision': 0.7257383966244726, 'negative_recall': 0.8911917098445595, 'negative_f1': 0.8, 'mixed_or_unclear_precision': 0.7683982683982684, 'mixed_or_unclear_recall': 0.7521186440677966, 'mixed_or_unclear_f1': 0.7601713062098501}


## Reports

In [13]:
report = classification_report(
    val_label_ids,
    val_pred_ids,
    target_names=ALLOWED_LABELS,
    output_dict=True,
    zero_division=0,
)

safe_json_dump(report, CLASSIFICATION_REPORT_JSON)
print("wrote:", CLASSIFICATION_REPORT_JSON)

cm = confusion_matrix(val_label_ids, val_pred_ids, labels=list(range(len(ALLOWED_LABELS))))
cm_df = pd.DataFrame(
    cm,
    index=[f"true_{x}" for x in ALLOWED_LABELS],
    columns=[f"pred_{x}" for x in ALLOWED_LABELS]
)
cm_df.to_csv(CONFUSION_MATRIX_CSV, index=True)
print("wrote:", CONFUSION_MATRIX_CSV)

report_by_type = {}
for entity_type, sub in valid_pred.groupby("final_type", dropna=False):
    if len(sub) < 20:
        continue
    sub_report = classification_report(
        sub["label_id"],
        sub["pred_id"],
        target_names=ALLOWED_LABELS,
        output_dict=True,
        zero_division=0,
    )
    report_by_type[str(entity_type)] = sub_report

safe_json_dump(report_by_type, CLASSIFICATION_REPORT_BY_TYPE_JSON)
print("wrote:", CLASSIFICATION_REPORT_BY_TYPE_JSON)

wrote: /content/drive/MyDrive/NLP_FINAL_PROJECT_Tom_Chen/output/05_sentiment_model/classification_report.json
wrote: /content/drive/MyDrive/NLP_FINAL_PROJECT_Tom_Chen/output/05_sentiment_model/confusion_matrix.csv
wrote: /content/drive/MyDrive/NLP_FINAL_PROJECT_Tom_Chen/output/05_sentiment_model/classification_report_by_type.json


## Export Predictions and Hard Cases

In [14]:
valid_pred = valid_pred.sort_values(
    ["is_correct", "pred_confidence"],
    ascending=[True, True]
).reset_index(drop=True)

valid_pred.to_parquet(VALIDATION_PREDICTIONS_PARQUET, index=False)
print("wrote:", VALIDATION_PREDICTIONS_PARQUET)

hard_cases = valid_pred[
    (~valid_pred["is_correct"])
].copy()

hard_cases = hard_cases.sort_values(
    ["pred_confidence", "final_type", "true_label", "pred_label"],
    ascending=[False, True, True, True]
).head(HARD_CASE_TOP_N).reset_index(drop=True)

hard_cases.to_parquet(VALIDATION_HARD_CASES_PARQUET, index=False)
print("wrote:", VALIDATION_HARD_CASES_PARQUET)

wrote: /content/drive/MyDrive/NLP_FINAL_PROJECT_Tom_Chen/output/05_sentiment_model/validation_predictions.parquet
wrote: /content/drive/MyDrive/NLP_FINAL_PROJECT_Tom_Chen/output/05_sentiment_model/validation_hard_cases.parquet


## Quick Readout

In [15]:
train_preview = pd.read_parquet(TRAIN_SPLIT_PARQUET)
valid_preview = pd.read_parquet(VALID_SPLIT_PARQUET)
pred_preview = pd.read_parquet(VALIDATION_PREDICTIONS_PARQUET)

print("train_split shape:", train_preview.shape)
print("valid_split shape:", valid_preview.shape)
print("validation_predictions shape:", pred_preview.shape)

print("\ntrain label distribution:")
display(train_preview["sentiment_label"].value_counts(dropna=False))

print("\nvalid label distribution:")
display(valid_preview["sentiment_label"].value_counts(dropna=False))

print("\nvalidation true label distribution:")
display(pred_preview["true_label"].value_counts(dropna=False))

print("\nvalidation predicted label distribution:")
display(pred_preview["pred_label"].value_counts(dropna=False))

print("\nvalidation accuracy by entity type:")
display(
    pred_preview.groupby("final_type", dropna=False)
    .agg(
        n_rows=("sample_id", "size"),
        accuracy=("is_correct", "mean"),
        confidence_mean=("pred_confidence", "mean"),
    )
    .reset_index()
    .sort_values(["n_rows", "accuracy"], ascending=[False, False])
)

print("\nworst confusion pairs:")
display(
    pred_preview[pred_preview["true_label"] != pred_preview["pred_label"]]
    .groupby(["true_label", "pred_label"], dropna=False)
    .agg(n_rows=("sample_id", "size"))
    .reset_index()
    .sort_values("n_rows", ascending=False)
    .head(20)
)

print("\nexample hard cases:")
display(
    pred_preview[pred_preview["true_label"] != pred_preview["pred_label"]][[
        "canonical_entity",
        "final_type",
        "true_label",
        "pred_label",
        "pred_confidence",
        "domain",
        "title",
        "context_text",
    ]].head(20)
)

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

train_split shape: (7251, 25)
valid_split shape: (1625, 25)
validation_predictions shape: (1625, 33)

train label distribution:


,count
sentiment_label,
positive,4135
mixed_or_unclear,2160
negative,956



valid label distribution:


,count
sentiment_label,
positive,960
mixed_or_unclear,472
negative,193



validation true label distribution:


...

,final_type,n_rows,accuracy,confidence_mean
0,company,618,0.870550,0.874479
3,technology,557,0.859964,0.867761
1,government_institution,230,0.852174,0.858211
2,person,220,0.795455,0.864412



worst confusion pairs:


,true_label,pred_label,n_rows
4,positive,mixed_or_unclear,87
1,mixed_or_unclear,positive,64
0,mixed_or_unclear,negative,53
2,negative,mixed_or_unclear,20
5,positive,negative,12
3,negative,positive,1



example hard cases:


,canonical_entity,final_type,true_label,pred_label,pred_confidence,domain,title,context_text
0,Howard Lutnick,person,mixed_or_unclear,positive,0.411515,devdiscourse.com,Commerce Nominee's Tough Stance on Trade and A...,Commerce Nominee's Tough Stance on Trade and A...
1,Sora 2,technology,negative,positive,0.412586,tolerance.ca,Tolerance.ca® - OpenAI’s newly launched Sora 2...,OpenAI’s newly launched Sora 2 makes AI’s envi...
2,Deepfake,technology,positive,negative,0.450264,zawya.com,931 tertiary education scholars trained in Art...,“The second reason is that if they know deepfa...
3,Cloud Computing,technology,negative,mixed_or_unclear,0.455395,breitbart.com,A Single AI Data Center Could Soon Consume Twi...,The rapid growth of AI and cloud computing is ...
4,Wi-Fi,technology,mixed_or_unclear,positive,0.467479,stuff.tv,Volumio Integro streamer uses AI to choose you...,There are plenty of ways to get a tune out of ...
5,Blackstone,company,mixed_or_unclear,positive,0.472761,whyy.org,Should we welcome AI data centers? - WHYY,"With the artificial intelligence boom, tech co..."
6,ING,company,positive,mixed_or_unclear,0.478331,einpresswire.com,CybeReady Provides Cybersecurity Awareness Mon...,About CybeReady CybeReady offers the world’s m...
7,Advanced Micro Devices Inc,company,positive,mixed_or_unclear,0.484286,nasdaq.com,Capture the Super-Cycle of AI Through IGPT | N...,“I’m very optimistic over the course of the ne...
8,Huang,person,mixed_or_unclear,positive,0.493508,nbcchicago.com,China’s DeepSeek quietly releases upgraded R1 ...,"""The U.S. has based its policy on the assumpti..."
9,Connecticut,government_institution,positive,mixed_or_unclear,0.493550,citylife.capetown,State Lawmakers Addressing Artificial Intellig...,State lawmakers in the United States are activ...
